# Machine Learning Assignment 2 — Dry Bean Classification

**Purpose:** Learn and run the complete classification workflow in BITS Virtual Lab.

Before submission:
- Read the code and comments; be ready to explain each step.
- Run the notebook in BITS Virtual Lab and inspect the actual outputs.
- Add your own observations after reviewing the metrics.
- Do not report results you have not personally generated and checked.

Dataset: UCI Dry Bean Dataset. Target column: `Class`.

In [ ]:
# Cell 1: Import libraries
import os
import zipfile
import warnings
import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
print("Imports successful")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

In [ ]:
# Cell 2: Download and extract the UCI dataset if it is not already present
dataset_file = "dataset/DryBeanDataset/Dry_Bean_Dataset.xlsx"

if not os.path.exists(dataset_file):
    os.makedirs("dataset", exist_ok=True)
    url = "https://archive.ics.uci.edu/static/public/602/dry+bean+dataset.zip"
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    with open("dry_bean.zip", "wb") as f:
        f.write(response.content)
    with zipfile.ZipFile("dry_bean.zip", "r") as zf:
        zf.extractall("dataset")

    # Locate the Excel file robustly, regardless of the extracted subfolder name
    matches = []
    for root, dirs, files in os.walk("dataset"):
        for name in files:
            if name.lower().endswith(".xlsx") and "dry" in name.lower():
                matches.append(os.path.join(root, name))
    if not matches:
        raise FileNotFoundError("Could not find the Dry Bean Excel file after extraction.")
    dataset_file = matches[0]

print("Dataset file:", dataset_file)

In [ ]:
# Cell 3: Load dataset and inspect it
df = pd.read_excel(dataset_file)
print("Original shape:", df.shape)
display(df.head())
print("\nDataset information:")
df.info()

In [ ]:
# Cell 4: Data quality checks
print("Missing values in dataset:", int(df.isnull().sum().sum()))
print("Duplicate rows before cleaning:", int(df.duplicated().sum()))
print("\nTarget class counts:")
display(df["Class"].value_counts())
print("Number of classes:", df["Class"].nunique())

In [ ]:
# Cell 5: Visualize target class distribution
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="Class", order=df["Class"].value_counts().index)
plt.title("Distribution of Dry Bean Classes")
plt.xlabel("Bean class")
plt.ylabel("Number of records")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Remove exact duplicate rows and separate features/target
df = df.drop_duplicates().reset_index(drop=True)

X = df.drop(columns=["Class"])
y = df["Class"]

print("Shape after duplicate removal:", df.shape)
print("Features (X):", X.shape)
print("Target (y):", y.shape)
print("Remaining duplicates:", int(df.duplicated().sum()))

In [ ]:
# Cell 7: Stratified 80:20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

In [ ]:
# Cell 8: Define five classification models.
# Scaling is included in a Pipeline for every model so the same saved object
# performs preprocessing and prediction consistently. Decision trees do not
# require scaling, but including it does not prevent the tree from working.
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(n_neighbors=5))
    ]),
    "Gaussian Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", GaussianNB())
    ]),
    "Support Vector Machine": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", SVC(probability=True, random_state=RANDOM_STATE))
    ])
}
print("Models defined:", list(models.keys()))

In [ ]:
# Cell 9: Train models and calculate all six required evaluation metrics
results = []
trained_models = {}

for model_name, model in models.items():
    print("Training:", model_name)
    model.fit(X_train, y_train)
    trained_models[model_name] = model

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    metrics = {
        "ML Model Name": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob, multi_class="ovr", average="macro"),
        "Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }
    results.append(metrics)

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
display(results_df.round(4))

In [ ]:
# Cell 10: Visualize model metric comparison
metrics_to_plot = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]
results_plot = results_df.set_index("ML Model Name")[metrics_to_plot]
ax = results_plot.plot(kind="bar", figsize=(13, 6))
ax.set_title("Comparison of Five Classification Models")
ax.set_ylabel("Score")
ax.set_ylim(-1, 1.05)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: Confusion matrix and classification report for each model
class_labels = sorted(y.unique())

for model_name, model in trained_models.items():
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=class_labels)

    print("\n" + "=" * 70)
    print(model_name)
    print(classification_report(y_test, y_pred, zero_division=0))

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_labels, yticklabels=class_labels)
    plt.title(f"Confusion Matrix — {model_name}")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 12: Save all five complete pipelines (including their scalers)
os.makedirs("model", exist_ok=True)

model_file_names = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "KNN": "knn.joblib",
    "Gaussian Naive Bayes": "naive_bayes.joblib",
    "Support Vector Machine": "svm.joblib"
}

for model_name, model in trained_models.items():
    joblib.dump(model, os.path.join("model", model_file_names[model_name]))

# Save the test set including its target column, as required for evaluation
test_data = X_test.copy()
test_data["Class"] = y_test.values
test_data.to_csv("test_data.csv", index=False)

# Save the actual metric table for README/report preparation
results_df.to_csv("model_comparison.csv", index=False)

print("Saved model files:", os.listdir("model"))
print("Saved test_data.csv:", os.path.exists("test_data.csv"))
print("Saved model_comparison.csv:", os.path.exists("model_comparison.csv"))

In [ ]:
# Cell 13: Show the actual winner based on macro F1 (do not assume in advance)
winner = results_df.iloc[0]
print("Winner by macro F1:", winner["ML Model Name"])
print("Macro F1:", round(float(winner["F1"]), 4))
print("\nUse the measured table and confusion matrices to write your own observations.")